# Getting started with TUICast

This notebook uses the Python SDK to reproduce the successful and unsuccessful login workflows from [`examples/features/login.feature`](../../features/login.feature). It launches `tuicast-driver`, connects to the reference TUI over SSH, and interacts with two independent terminal sessions.

Before running the notebook, install the SDK and Jupyter in a Python 3.12+ environment. From the repository root:

```sh
python3.12 -m venv sdk/python/.venv
sdk/python/.venv/bin/python -m pip install -e sdk/python jupyterlab
```


## Start the reference TUI

Build the driver and reference application, then start the reference application in a separate terminal:

```sh
mkdir -p /tmp/tuicast-example
go build -o /tmp/tuicast-example/tuicast-driver ./cmd/tuicast-driver
go build -o /tmp/tuicast-example/reference-tui ./cmd/reference-tui
/tmp/tuicast-example/reference-tui --ssh-address 127.0.0.1:2222
```

Then launch this notebook from the repository root:

```sh
TUICAST_DRIVER=/tmp/tuicast-example/tuicast-driver \
  sdk/python/.venv/bin/python -m jupyter lab examples/python/notebooks/getting_started.ipynb
```

The reference fixture uses `operator` / `casting` for both its SSH and application credentials. Host-key checking is disabled below only because this is a local deterministic fixture.


## Connect to the driver and reference TUI

The SDK starts the driver as a child process. The driver owns the SSH connection and terminal emulation. Passing `-ui-address` also starts the read-only TUICast Inspector at [http://127.0.0.1:8080](http://127.0.0.1:8080), where you can watch connections and sessions appear as you run the cells. Keeping each resource in a variable lets us inspect and close it explicitly as we work through the notebook.


In [ ]:
import os

import tuicast

driver = tuicast.Driver.launch(
    os.getenv("TUICAST_DRIVER", "tuicast-driver"),
    args=("-ui-address", "127.0.0.1:8080"),
)

connection = driver.connect(
    tuicast.SSH(
        os.getenv("TUICAST_REFERENCE_ADDRESS", "127.0.0.1:2222"),
        "operator",
        password="casting",
        insecure_skip_host_key_check=True,
    )
)
session = None


time=2026-09-27T20:38:15.838-04:00 level=INFO msg="TUICast Inspector listening" url=http://127.0.0.1:8080
time=2026-09-27T20:38:15.838-04:00 level=INFO msg="TUICast Workbench enabled" url=http://127.0.0.1:8080
time=2026-09-27T20:38:15.849-04:00 level=INFO msg="terminal connection opened" connection_id=1


## Successful login

Open an xterm-compatible session and wait for a stable login screen. `Screen.text` is a plain-text snapshot of the emulated terminal.


In [27]:
session = connection.open_session(
    terminal=tuicast.Terminal.XTERM_256COLOR
)
screen = session.wait_for_text(
    "LOGIN / AUTHENTICATION", stable_for=0.05
)
print(screen.text)


TUICAST REFERENCE TERMINAL                                                      
                                                                                
LOGIN / AUTHENTICATION                                                          
                                                                                
User ID:                                                                        
          ────────────────────────                                              
                                                                                
Password:                                                                       
          ────────────────────────                                              
                                                                                
F1 Login    F2 Clear    Ctrl-C Exit                                             
                                                                                
Test credentials: operator /

time=2026-09-27T20:43:34.690-04:00 level=INFO msg="terminal session opened" connection_id=1 session_id=4


Enter the documented application credentials. The SDK sends text and named keys separately, then waits against terminal state rather than raw network output.


In [28]:
session.type("operator")
session.press(tuicast.Key.TAB)
session.type("casting")
session.press(tuicast.Key.ENTER)

screen = session.wait_for_text(
    "TERMINAL TEST SYSTEM", stable_for=0.1
)
assert screen.contains("Authenticated as operator")
print(screen.text)


Password:                                                                       
          ────────────────────────                                              
                                                                                
F1 Login    F2 Clear    Ctrl-C Exit                                             
                                                                                
TERMINAL TEST SYSTEM                                                            
                                                                                
 1. Login / Authentication                                                      
 2. Forms and Input Fields                                                      
 3. Tables                                                                      
 4. Scrolling                                                                   
 5. Colors and Attributes                                                       
 6. Cursor Movement         

Close this session so the unsuccessful-login scenario starts with a fresh terminal, matching the scenario isolation in the feature file.


In [24]:
session.close()
session = None


time=2026-09-27T20:42:29.510-04:00 level=INFO msg="terminal session closed" connection_id=1 session_id=1


## Unsuccessful login

Open another session and submit invalid credentials. The application should remain on the login screen and report the authentication failure.


In [25]:
session = connection.open_session(
    terminal=tuicast.Terminal.XTERM_256COLOR
)
session.wait_for_text("LOGIN / AUTHENTICATION", stable_for=0.05)

session.type("unknown")
session.press(tuicast.Key.TAB)
session.type("incorrect")
session.press(tuicast.Key.ENTER)

screen = session.wait_for(
    tuicast.all_of(
        tuicast.contains("LOGIN / AUTHENTICATION"),
        tuicast.contains("Invalid user ID or password"),
    ),
    stable_for=0.1,
)
print(screen.text)


time=2026-09-27T20:42:36.795-04:00 level=INFO msg="terminal session opened" connection_id=1 session_id=2


TUICAST REFERENCE TERMINAL                                                      
                                                                                
LOGIN / AUTHENTICATION                                                          
                                                                                
User ID:  unknown                                                               
          ────────────────────────                                              
                                                                                
Password:                                                                       
          ────────────────────────                                              
                                                                                
F1 Login    F2 Clear    Ctrl-C Exit                                             
                                                                                
Test credentials: operator /

## Close the resources

Always close resources in reverse ownership order: session, connection, then driver. The nested `finally` blocks ensure the driver process is stopped even if closing another resource raises an error. TUICast close operations are idempotent, so this cell is safe to run again.


In [29]:
try:
    if session is not None:
        session.close()
finally:
    try:
        if connection is not None:
            connection.close()
    finally:
        if driver is not None:
            driver.close()

session = None
connection = None
driver = None


time=2026-09-27T20:44:14.193-04:00 level=INFO msg="terminal session closed" connection_id=1 session_id=4
time=2026-09-27T20:44:14.194-04:00 level=INFO msg="terminal session closed" connection_id=1 session_id=2
time=2026-09-27T20:44:14.194-04:00 level=INFO msg="terminal session closed" connection_id=1 session_id=3
time=2026-09-27T20:44:14.194-04:00 level=INFO msg="terminal connection closed" connection_id=1
time=2026-09-27T20:44:14.194-04:00 level=INFO msg="terminal server closed"
